In [25]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, roc_auc_score
import joblib

In [5]:
d_labitems = pd.read_csv('../data/D_LABITEMS.csv')
d_items = pd.read_csv('../data/D_ITEMS.csv')

In [6]:
feature_groups = {
    # из D_ITEMS
    'heart_rate': ['heart rate', 'pulse'],
    'sys_bp': ['systolic', 'arterial bp systolic'],
    'dia_bp': ['diastolic', 'arterial bp diastolic'],
    'mean_bp': ['mean arterial pressure', 'arterial bp mean'],
    'resp_rate': ['respiratory rate', 'respiration'],
    'temperature': ['temperature'],
    'spo2': ['oxygen saturation', 'spo2'],
    
    # из D_LABITEMS
    'glucose': ['glucose'],
    'potassium': ['potassium'],
    'sodium': ['sodium'],
    'creatinine': ['creatinine'],
    'hematocrit': ['hematocrit'],
    'hemoglobin': ['hemoglobin'],
    'wbc': ['white blood cell', 'wbc'],
    'platelets': ['platelet'],
}


In [ ]:
#извлечени фич
def find_itemids(df, keywords, id_col='itemid', label_col='label'):
    mask = df[label_col].str.lower().str.contains('|'.join(keywords), na=False)
    return df.loc[mask, id_col].tolist()

feature_itemids = {}

for feature, keywords in feature_groups.items():
    itemids = []
    itemids += find_itemids(d_items, keywords)
    itemids += find_itemids(d_labitems, keywords)
    feature_itemids[feature] = list(set(itemids))

for feat, ids in feature_itemids.items():
    print(f"{feat}: {len(ids)}")

heart_rate: 61
sys_bp: 27
dia_bp: 26
mean_bp: 2
resp_rate: 7
temperature: 17
spo2: 7
glucose: 31
potassium: 47
sodium: 70
creatinine: 21
hematocrit: 15
hemoglobin: 14
wbc: 20
platelets: 14


In [ ]:
import numpy as np
from collections import defaultdict

def aggregate_events(filepath, feature_itemids, chunksize=100000):
    """
    собираем события из CSV-файла по HADM_ID и ITEMID
    результат: hadm_id -> {feature: {'values': [], 'units': []}}
    """
    result = defaultdict(lambda: defaultdict(list))
    
    for chunk in pd.read_csv(filepath, chunksize=chunksize, usecols=['hadm_id', 'itemid', 'valuenum', 'valueuom']):
        mask = chunk['itemid'].isin([item for sublist in feature_itemids.values() for item in sublist])
        filtered = chunk[mask].dropna(subset=['valuenum'])
        
        for _, row in filtered.iterrows():
            hadm = row['hadm_id']
            itemid = row['itemid']
            value = row['valuenum']
            for feature, ids in feature_itemids.items():
                if itemid in ids:
                    result[hadm][feature].append(value)
                    break
    return result

lab_data = aggregate_events('../data/LABEVENTS.csv', feature_itemids)
chart_data = aggregate_events('../data/CHARTEVENTS.csv', feature_itemids)

In [9]:
all_hadm_ids = set(lab_data.keys()) | set(chart_data.keys())

X_rows = []
for hadm in all_hadm_ids:
    row = {'hadm_id': hadm}
    combined = defaultdict(list)
    if hadm in lab_data:
        for feature, values in lab_data[hadm].items():
            combined[feature].extend(values)
    if hadm in chart_data:
        for feature, values in chart_data[hadm].items():
            combined[feature].extend(values)
    
    for feature, values in combined.items():
        if values:
            row[f'{feature}_mean'] = np.mean(values)
            row[f'{feature}_min'] = np.min(values)
            row[f'{feature}_max'] = np.max(values)
            row[f'{feature}_count'] = len(values)
        else:
            row[f'{feature}_mean'] = np.nan
            row[f'{feature}_min'] = np.nan
            row[f'{feature}_max'] = np.nan
            row[f'{feature}_count'] = 0
    X_rows.append(row)

X_df = pd.DataFrame(X_rows)
X_df

,hadm_id,platelets_mean,platelets_min,platelets_max,platelets_count,potassium_mean,potassium_min,potassium_max,potassium_count,wbc_mean,...,sys_bp_max,sys_bp_count,dia_bp_mean,dia_bp_min,dia_bp_max,dia_bp_count,mean_bp_mean,mean_bp_min,mean_bp_max,mean_bp_count
0,NaN,42.0,42.0,42.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,3.8,3.8,3.8,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4031,NaN,111.0,111.0,111.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4032,NaN,NaN,NaN,NaN,NaN,3.9,3.9,3.9,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#добавление демографических признаков
admissions = pd.read_csv('../data/ADMISSIONS.csv', 
                         usecols=['hadm_id', 'subject_id', 'admittime', 'admission_type', 'hospital_expire_flag'])
patients = pd.read_csv('../data/PATIENTS.csv', usecols=['subject_id', 'gender', 'dob'])
X_df = X_df.merge(admissions, on='hadm_id', how='left')
X_df = X_df.merge(patients, on='subject_id', how='left')

X_df['admittime'] = pd.to_datetime(X_df['admittime'])
X_df['dob'] = pd.to_datetime(X_df['dob'])
X_df['admission_year'] = X_df['admittime'].dt.year
X_df['birth_year'] = X_df['dob'].dt.year

X_df['age'] = X_df['admission_year'] - X_df['birth_year']
X_df = X_df[(X_df['age'] >= 0) & (X_df['age'] <= 120)]
X_df = X_df.drop(columns=['dob', 'admittime', 'admission_year', 'birth_year'])
X_df['gender'] = (X_df['gender'] == 'M').astype(int)

print(f"Размер X_df после добавления демографических признаков: {X_df.shape}")
print(X_df.head())

Размер X_df после добавления демографических признаков: (120, 66)
      hadm_id  platelets_mean  platelets_min  platelets_max  platelets_count  \
47   172082.0      222.800000          169.0          264.0             25.0   
155  188574.0      194.800000          180.0          212.0             10.0   
411  180685.0      237.000000          229.0          241.0              3.0   
490  139932.0      273.533333          163.0          380.0             60.0   
565  197611.0      381.461538          210.0          493.0             26.0   

     potassium_mean  potassium_min  potassium_max  potassium_count   wbc_mean  \
47         4.430556            3.2           31.0             36.0  13.834615   
155        3.480000            3.2            3.9             10.0  13.327778   
411        4.500000            3.1            5.0             16.0  21.683333   
490        3.661250            2.6            5.0             80.0   7.380000   
565        3.837931            3.0            4.

In [11]:
diagnoses = pd.read_csv('../data/DIAGNOSES_ICD.csv', usecols=['hadm_id', 'icd9_code'])
target_hadm = set(diagnoses['hadm_id'].unique())
print(f"Всего госпитализаций с диагнозами: {len(target_hadm)}")

Всего госпитализаций с диагнозами: 129


In [12]:
import numpy as np
from collections import defaultdict

def aggregate_events_for_target(filepath, target_hadm, feature_itemids, chunksize=100000):
    """
    события только для заданных hadm_id
    на выходе словарь: hadm_id -> {feature: список значений}
    """
    result = defaultdict(lambda: defaultdict(list))
    
    for chunk in pd.read_csv(filepath, chunksize=chunksize,
                             usecols=['hadm_id', 'itemid', 'valuenum']):
        chunk = chunk[chunk['hadm_id'].isin(target_hadm)]
        if chunk.empty:
            continue
        all_itemids = [item for sublist in feature_itemids.values() for item in sublist]
        mask = chunk['itemid'].isin(all_itemids)
        filtered = chunk[mask].dropna(subset=['valuenum'])
        
        for _, row in filtered.iterrows():
            hadm = row['hadm_id']
            itemid = row['itemid']
            value = row['valuenum']
            for feature, ids in feature_itemids.items():
                if itemid in ids:
                    result[hadm][feature].append(value)
                    break
    return result

In [13]:
lab_data = aggregate_events_for_target('../data/LABEVENTS.csv', target_hadm, feature_itemids)
chart_data = aggregate_events_for_target('../data/CHARTEVENTS.csv', target_hadm, feature_itemids)

print(f"Найдено данных в LABEVENTS для {len(lab_data)} госпитализаций")
print(f"Найдено данных в CHARTEVENTS для {len(chart_data)} госпитализаций")

Найдено данных в LABEVENTS для 129 госпитализаций
Найдено данных в CHARTEVENTS для 125 госпитализаций


In [14]:
X_rows = []
all_hadm = target_hadm

for hadm in all_hadm:
    row = {'hadm_id': hadm}
    combined = defaultdict(list)
    
    if hadm in lab_data:
        for feature, values in lab_data[hadm].items():
            combined[feature].extend(values)
    if hadm in chart_data:
        for feature, values in chart_data[hadm].items():
            combined[feature].extend(values)
    for feature, values in combined.items():
        if values:
            row[f'{feature}_mean'] = np.mean(values)
            row[f'{feature}_min'] = np.min(values)
            row[f'{feature}_max'] = np.max(values)
            row[f'{feature}_count'] = len(values)
        else:
            row[f'{feature}_mean'] = np.nan
            row[f'{feature}_min'] = np.nan
            row[f'{feature}_max'] = np.nan
            row[f'{feature}_count'] = 0
    X_rows.append(row)

X_df = pd.DataFrame(X_rows)
X_df = X_df.set_index('hadm_id')
print(f"Размер X_df: {X_df.shape}")

Размер X_df: (129, 60)


In [15]:
admissions = pd.read_csv('../data/ADMISSIONS.csv', usecols=['hadm_id', 'subject_id', 'admittime', 'admission_type', 'hospital_expire_flag'])
patients = pd.read_csv('../data/PATIENTS.csv', usecols=['subject_id', 'gender', 'dob'])

X_df = X_df.reset_index().merge(admissions, on='hadm_id', how='left').set_index('hadm_id')
X_df = X_df.reset_index().merge(patients, on='subject_id', how='left').set_index('hadm_id')

In [16]:
X_df['admittime'] = pd.to_datetime(X_df['admittime'])
X_df['dob'] = pd.to_datetime(X_df['dob'])
X_df['age'] = X_df['admittime'].dt.year - X_df['dob'].dt.year
X_df = X_df.drop(columns=['dob', 'admittime', 'subject_id'])
X_df['gender'] = (X_df['gender'] == 'M').astype(int)

print(f"Размер X_df после добавления демографии: {X_df.shape}")

Размер X_df после добавления демографии: (129, 64)


In [17]:
hadm_codes = diagnoses.groupby('hadm_id')['icd9_code'].apply(list).reset_index()
hadm_codes = hadm_codes[hadm_codes['hadm_id'].isin(X_df.index)]

mlb = MultiLabelBinarizer()
y_full = mlb.fit_transform(hadm_codes['icd9_code'])
class_names_full = mlb.classes_

In [18]:
code_counts = y_full.sum(axis=0)
top_k = 50
top_indices = np.argsort(code_counts)[-top_k:]
y = y_full[:, top_indices]
class_names = class_names_full[top_indices]

X_df = X_df.loc[hadm_codes['hadm_id']]
y_df = pd.DataFrame(y, index=hadm_codes['hadm_id'], columns=class_names)
y = y_df.values

print(f"X shape: {X_df.shape}, y shape: {y.shape}")

X shape: (129, 64), y shape: (129, 50)


In [19]:
X_train, X_temp, y_train, y_temp = train_test_split(X_df, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(f"train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")

train: (90, 64), val: (19, 64), test: (20, 64)


In [20]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"num features: {len(numeric_features)}")

imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train[numeric_features])
X_val_imp = imputer.transform(X_val[numeric_features])
X_test_imp = imputer.transform(X_test[numeric_features])

num features: 63


In [21]:
X_train_imp = pd.DataFrame(X_train_imp, columns=numeric_features, index=X_train.index)
X_val_imp = pd.DataFrame(X_val_imp, columns=numeric_features, index=X_val.index)
X_test_imp = pd.DataFrame(X_test_imp, columns=numeric_features, index=X_test.index)

In [30]:
base_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

multi_model = MultiOutputClassifier(base_model, n_jobs=-1)
pos_freq = y_train.mean(axis=0)
pos_freq = np.where(pos_freq == 0, 1, pos_freq)
weights_per_label = 1.0 / np.sqrt(pos_freq)
sample_weights = np.array([
    np.mean(weights_per_label[y_train[i] == 1]) if (y_train[i] == 1).any() else 1.0
    for i in range(y_train.shape[0])
])
multi_model.fit(X_train_imp, y_train, sample_weight=sample_weights)

/opt/homebrew/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [19:06:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/homebrew/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [19:06:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/homebrew/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [19:06:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/homebrew/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [19:06:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fo

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,"XGBClassifier...ree=None, ...)"
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",-1
,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None


In [31]:
y_val_proba = np.array([est.predict_proba(X_val_imp)[:, 1] for est in multi_model.estimators_]).T
y_val_pred = (y_val_proba > 0.5).astype(int)

f1_micro = f1_score(y_val, y_val_pred, average='micro')
f1_macro = f1_score(y_val, y_val_pred, average='macro')
print(f"F1 micro: {f1_micro:.4f}")
print(f"F1 macro: {f1_macro:.4f}")

aucs = []
for i in range(y_val.shape[1]):
    if len(np.unique(y_val[:, i])) > 1:
        aucs.append(roc_auc_score(y_val[:, i], y_val_proba[:, i]))
print(f"Средний ROC-AUC: {np.mean(aucs):.4f}")

F1 micro: 0.2400
F1 macro: 0.1234
Средний ROC-AUC: 0.6252


/opt/homebrew/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [32]:
joblib.dump(imputer, '../models/imputer.pkl')
joblib.dump(numeric_features, '../models/numeric_features.pkl')
joblib.dump(multi_model, '../models/multi_model.pkl')
joblib.dump(mlb, '../models/mlb.pkl')
joblib.dump(feature_itemids, '../models/feature_itemids.pkl')
joblib.dump(class_names, '../models/class_names.pkl')

['../models/class_names.pkl']